## Data Processing Notebook

In [1]:
import os

from sqlalchemy import create_engine, text
import pandas as pd

import sys
sys.path.append('../..')

from final_project.scripts.data_processing_utils import setup_categorical_type

In [2]:
rurality_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_attainment_by_rurality.csv"))
chars_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_attainment_by_characteristics.csv"))
retention_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_retention_by_region.csv"))
results_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_results_by_subject.csv"))
stem_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_stem_by_sex.csv"))

In [8]:
with open(rurality_path, "r") as f:
    rurality_df = pd.read_csv(f)

In [19]:
rurality_df = setup_categorical_type(rurality_df, "aps_per_entry_grade_acad")
rurality_df = setup_categorical_type(rurality_df, "aps_per_entry_grade_alev")

rurality_df.describe()

,time_period,number_of_students_acad,number_of_students_alev,number_of_students_potential,pc_achieving_atleast_two_alev,pc_achieving_3_astar_to_a_alev
count,315.000000,315.000000,315.000000,315.000000,215.000000,214.000000
mean,202122.000000,4022.898413,3943.996825,8701.946032,84.674088,16.821313
std,143.062834,8571.439986,8395.876514,17481.073505,9.316256,6.976126
min,201920.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,202021.000000,0.000000,0.000000,0.000000,82.726000,12.500000
50%,202122.000000,226.000000,224.000000,836.000000,86.312000,16.808000
75%,202223.000000,3535.500000,3512.000000,7177.000000,88.500500,21.254500
max,202324.000000,48371.000000,47392.000000,87284.000000,100.000000,40.000000


In [4]:
engine = create_engine('sqlite:///../data/database.db')

In [6]:
ees_attainment_by_rurality_schema = """
CREATE TABLE ees_attainment_by_rurality (
    time_period VARCHAR(10),
    version VARCHAR(20),
    region_name VARCHAR(50),
    rurality_name VARCHAR(50),
    number_of_students_acad INTEGER,
    number_of_students_alev INTEGER,
    number_of_students_potential INTEGER,
    aps_per_entry_grade_acad VARCHAR(5),
    aps_per_entry_grade_alev VARCHAR(5),
    pc_achieving_atleast_two_alev DECIMAL(5,2),
    pc_achieving_3_astar_to_a_alev DECIMAL(5,2)
)
"""

ees_attainment_by_characteristics_schema = """
CREATE TABLE ees_attainment_by_characteristics (
    time_period VARCHAR(10),
    version VARCHAR(20),
    region_name VARCHAR(50),
    characteristic_type VARCHAR(50),
    characteristic_value VARCHAR(50),
    establishment_type VARCHAR(100),
    number_of_students_acad INTEGER,
    number_of_students_alev INTEGER,
    number_of_students_potential INTEGER,
    aps_per_entry_grade_acad VARCHAR(5),
    aps_per_entry_grade_alev VARCHAR(5),
    pc_achieving_atleast_two_alev DECIMAL(5,2),
    pc_achieving_3_astar_to_a_alev DECIMAL(5,2)
)
"""

ees_retention_by_region_schema = """
CREATE TABLE ees_retention_by_region (
    time_period VARCHAR(10),
    version VARCHAR(20),
    region_name VARCHAR(50),
    geographic_level VARCHAR(20),
    qualification VARCHAR(50),
    cohort_size INTEGER,
    cohort_size_withdrawn INTEGER,
    retained_size INTEGER,
    retained_size_withdrawn INTEGER,
    retention_rate DECIMAL(6,3),
    retention_rate_withdrawn DECIMAL(6,3)
)
"""

ees_results_by_subject_schema = """
CREATE TABLE ees_results_by_subject (
    time_period VARCHAR(10),
    version VARCHAR(20),
    region_name VARCHAR(50),
    qualification VARCHAR(50),
    subject_name VARCHAR(100),
    geographic_level VARCHAR(20),
    subject_area VARCHAR(100),
    entry_count INTEGER,
    astar_grade_achieved INTEGER,
    a_grade_achieved INTEGER,
    b_grade_achieved INTEGER,
    c_grade_achieved INTEGER,
    d_grade_achieved INTEGER,
    e_grade_achieved INTEGER,
    u_grade_achieved INTEGER,
    astar_a_grade_achieved INTEGER,
    astar_b_grade_achieved INTEGER,
    astar_c_grade_achieved INTEGER,
    astar_d_grade_achieved INTEGER,
    astar_e_grade_achieved INTEGER
)
"""

ees_stem_by_sex_schema = """
CREATE TABLE ees_stem_by_sex (
    time_period VARCHAR(10),
    version VARCHAR(20),
    region_name VARCHAR(50),
    characteristic_sex VARCHAR(10),
    sub_name_comb VARCHAR(200),
    num_maths_science INTEGER,
    num_entered_comb_and_no_other_matsci INTEGER,
    num_entered_matsci_subjects_including_comb INTEGER
)
"""

with engine.connect() as conn:
    conn.execute(text(ees_attainment_by_rurality_schema))
    conn.execute(text(ees_attainment_by_characteristics_schema))
    conn.execute(text(ees_retention_by_region_schema))
    conn.execute(text(ees_results_by_subject_schema))
    conn.execute(text(ees_stem_by_sex_schema))

    conn.commit()



In [ ]:
rurality_df["dropout_rate"] = (rurality_df["number_of_students_entered"] - 
                               rurality_df["number_of_students_potential"])/ rurality_df["number_of_students_entered"]

rurality_df["pc_of_students_doing_alev"] = (rurality_df["number_of_students_alev"] / rurality_df["number_of_students_entered"]) * 100

In [ ]:
chars_df.groupby("region_name").apply(lambda x: x.isnull().sum(), include_groups=False)

In [ ]:
chars_df[chars_df["number_of_students_potential"].isnull()]

In [ ]:
chars_df["characteristic_type"].unique()

In [ ]:
grouped = chars_df[(chars_df["characteristic_type"]=="All Students")].groupby(
    ["region_name"]).agg({
    "number_of_students_acad":"sum",
    "number_of_students_alev": "sum",
    "number_of_students_potential": "sum",
    "aps_per_entry_grade_acad": lambda x: x.mode()[0],
    "aps_per_entry_grade_alev": lambda x: x.mode()[0]
    })

total_row = grouped.sum(numeric_only=True)
total_row.name = 'Total'
grouped_with_total = pd.concat([grouped, total_row.to_frame().T])

grouped_with_total

In [12]:
engine.dispose()